In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/whisper_konkani/konkani_whisper_dataset_split"))


True


In [ ]:
base_dir = "/content/drive/MyDrive/whisper_konkani"
dataset_path = f"{base_dir}/konkani_whisper_dataset_split"
tokenizer_path = f"{base_dir}/konkani-bpe-tokenizer"


In [ ]:
!pip install transformers datasets torchaudio accelerate


In [ ]:
from datasets import load_from_disk
from transformers import WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from transformers import PreTrainedTokenizerFast
import torch

# Paths
dataset_path = "file:///content/drive/MyDrive/whisper_konkani/konkani_whisper_dataset_split"
tokenizer_path = "/content/drive/MyDrive/whisper_konkani/konkani-bpe-tokenizer"

# 1. Load dataset
dataset = load_from_disk(dataset_path)

# 2. Load tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(tokenizer_path)

# 3. Load Whisper model and processor
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

# 4. Preprocessing
def preprocess(batch):
    audio = batch["audio"]
    input_features = processor.feature_extractor(audio["array"], sampling_rate=16000, return_tensors="pt").input_features[0]
    labels = tokenizer(batch["transcription"], padding="max_length", max_length=128, truncation=True).input_ids
    return {
        "input_features": input_features,
        "labels": labels
    }

dataset = dataset.map(preprocess, remove_columns=dataset["train"].column_names, num_proc=2)

# 5. Training args
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/whisper-konkani-finetuned",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    warmup_steps=500,
    num_train_epochs=3,
    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    predict_with_generate=True,
    fp16=True,
)

# 6. Trainer
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    tokenizer=processor,
)

# 7. Train
trainer.train()

# 8. Save model
model.save_pretrained("/content/drive/MyDrive/whisper_konkani/whisper-konkani-finetuned")
processor.save_pretrained("/content/drive/MyDrive/whisper_konkani/whisper-konkani-finetuned")

/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
/usr/local/lib/python3.11/dist-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)
<ipython-input-9-1053034397>:50: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: brtpesu2027 (brtpesu2027-pes-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.43.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


Step,Training Loss


In [ ]:
!ls /content/drive/MyDrive/whisper_konkani


konkani-bpe-tokenizer  konkani_whisper_dataset_split
